# 02 · Limpieza de datos — IEEE-CIS Fraud Detection

**Objetivo:** dejar un dataset limpio (sin missing, sin categóricas en texto) listo para feature
engineering, **sin fugas de información**. Este notebook es autocontenido: vuelve a cargar y
mergear los datos crudos en vez de depender de variables dejadas en memoria por `01_exploracion.ipynb`.

Regla que se respeta en todo el notebook: **el split train/test ocurre primero**, y cualquier
parámetro de limpieza (umbral de missing, medianas, categorías frecuentes) se calcula **solo con
el train** y se aplica igual al test.

In [1]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
import joblib

warnings.filterwarnings("ignore")
sys.path.append(str(Path("..").resolve()))
from src.utils import reduce_mem_usage

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
ART_DIR = PROC_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


## 1. Carga y merge (idéntico a 01_exploracion.ipynb)

In [2]:
train_transaction = reduce_mem_usage(pd.read_csv(RAW_DIR / "train_transaction.csv"))
train_identity = reduce_mem_usage(pd.read_csv(RAW_DIR / "train_identity.csv"))
df = train_transaction.merge(train_identity, on="TransactionID", how="left")
del train_transaction, train_identity
print("df:", df.shape)


Memoria: 1791.67 MB -> 861.11 MB (reducción del 51.9%)


Memoria: 56.51 MB -> 16.01 MB (reducción del 71.7%)


df: (590540, 434)


## 2. Split estratificado train/test (ANTES de imputar o codificar nada)

Se separa un 20% de las transacciones como test, estratificando por `isFraud` para que ambos
conjuntos conserven la misma proporción de fraude (~3.5%). A partir de aquí, **todo parámetro
aprendido (medianas, categorías frecuentes, encoders) se ajusta solo con `train`**.

In [3]:
y = df["isFraud"]
X = df.drop(columns=["isFraud"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
del df, X, y

print("X_train:", X_train.shape, "| tasa de fraude:", y_train.mean().round(4))
print("X_test: ", X_test.shape, "| tasa de fraude:", y_test.mean().round(4))


X_train: (472432, 433) | tasa de fraude: 0.035
X_test:  (118108, 433) | tasa de fraude: 0.035


## 3. Eliminación de columnas con demasiados valores faltantes

**Umbral: >90% de missing.** Se eligió este umbral (sugerido en el EDA de `01_exploracion.ipynb`,
donde se identificó un grupo grande de columnas `id_*` y `V*` con missing extremo) porque una
columna con más del 90% de valores faltantes aporta señal real a menos del 10% de las filas: aun
imputándola, el riesgo de introducir ruido supera el beneficio de la poca información real que
queda. El % de missing se calcula **solo con `X_train`** y la misma lista de columnas se elimina
en `X_test`.

In [4]:
missing_pct_train = X_train.isna().mean()
cols_to_drop = missing_pct_train[missing_pct_train > 0.90].index.tolist()

print(f"Columnas eliminadas por >90% missing en train: {len(cols_to_drop)} de {X_train.shape[1]}")
print(cols_to_drop[:15], "..." if len(cols_to_drop) > 15 else "")

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)
print("Nueva forma -> X_train:", X_train.shape, "| X_test:", X_test.shape)


Columnas eliminadas por >90% missing en train: 12 de 433
['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27'] 
Nueva forma -> X_train: (472432, 421) | X_test: (118108, 421)


## 4. Separar columnas numéricas y categóricas

`reduce_mem_usage` ya convirtió las columnas de texto de baja cardinalidad a `category`. Usamos
esa distinción de dtype para separar el pipeline de imputación/encoding.

In [5]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()

print(f"Columnas categóricas: {len(cat_cols)}")
print(cat_cols)
print(f"\nColumnas numéricas: {len(num_cols)}")


Columnas categóricas: 29
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

Columnas numéricas: 392


## 5. Imputación de valores faltantes

- **Numéricas:** mediana calculada en `train` (`SimpleImputer(strategy="median")`). Se prefiere
  la mediana sobre la media porque varias columnas numéricas (`TransactionAmt`, `C*`, `D*`) están
  sesgadas y tienen outliers.
- **Categóricas:** se rellenan con la categoría explícita `"missing"` — no es un parámetro que se
  "aprenda" (es una constante), así que no hay riesgo de fuga, y preserva la señal de que el dato
  faltaba (lo cual puede ser informativo, p. ej. no tener `DeviceType` suele correlacionar con el
  canal de la transacción).

In [6]:
num_imputer = SimpleImputer(strategy="median")
num_imputer.set_output(transform="pandas")
num_imputer.fit(X_train[num_cols])

X_train[num_cols] = num_imputer.transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

for c in cat_cols:
    X_train[c] = X_train[c].astype("object").fillna("missing")
    X_test[c] = X_test[c].astype("object").fillna("missing")

print("Missing restante en X_train:", X_train.isna().sum().sum())
print("Missing restante en X_test:", X_test.isna().sum().sum())


Missing restante en X_train: 0


Missing restante en X_test: 0


## 6. Agrupar categorías raras (aprendido solo en train)

Columnas como `DeviceInfo` o `id_31` (navegador) tienen decenas o cientos de valores únicos, muchos
de ellos vistos una sola vez. Para evitar que el encoder aprenda categorías casi-únicas (que no
generalizan) y para manejar de forma consistente las categorías que solo aparecen en test,
agrupamos como `"rare"` cualquier categoría que represente menos del 0.1% de las filas de train.
Como la decisión de qué es "frecuente" se toma solo con train, cualquier categoría de test que no
esté en ese conjunto frecuente (ya sea porque es rara o porque nunca apareció en train) también se
mapea a `"rare"` — así no quedan categorías nuevas sin resolver antes del encoding.

In [7]:
RARE_THRESHOLD = 0.001  # 0.1% de las filas de train
frequent_categories = {}

for c in cat_cols:
    freq = X_train[c].value_counts(normalize=True)
    frequent_categories[c] = set(freq[freq >= RARE_THRESHOLD].index)
    X_train[c] = X_train[c].where(X_train[c].isin(frequent_categories[c]), "rare")
    X_test[c] = X_test[c].where(X_test[c].isin(frequent_categories[c]), "rare")

n_rare_train = (X_train[cat_cols] == "rare").sum().sum()
n_rare_test = (X_test[cat_cols] == "rare").sum().sum()
print(f"Celdas agrupadas como 'rare' -> train: {n_rare_train} | test: {n_rare_test}")


Celdas agrupadas como 'rare' -> train: 54693 | test: 13419


## 7. Codificación de variables categóricas

Se usa **ordinal/label encoding** (`OrdinalEncoder`) en vez de one-hot por dos razones prácticas:

1. Varias columnas categóricas (`DeviceInfo`, `id_31`, `id_33`) siguen teniendo decenas de
   categorías incluso después de agrupar las raras — one-hot dispararía la dimensionalidad.
2. Random Forest no se ve perjudicado por el orden arbitrario de los códigos (encuentra los
   splits igual). Para la regresión logística esto sí es una limitación real (asume un orden que
   no existe) — es un trade-off consciente: se documenta aquí y se compensa en
   `05_modelo.ipynb` con regularización; si se buscara exprimir más a la regresión logística se
   podría usar one-hot solo para las columnas categóricas de baja cardinalidad (`ProductCD`,
   `card4`, `card6`, `M1..M9`), pero se mantiene un único encoding para simplicidad y para que
   ambos modelos consuman exactamente el mismo dataset.

El encoder se ajusta **solo con `X_train`**; `handle_unknown="use_encoded_value"` con
`unknown_value=-1` es una red de seguridad adicional para cualquier categoría no vista.

In [8]:
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoder.fit(X_train[cat_cols])

X_train[cat_cols] = encoder.transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

X_train[cat_cols] = X_train[cat_cols].astype("int32")
X_test[cat_cols] = X_test[cat_cols].astype("int32")

X_train.head()


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
40809,3027809.0,1008491.0,100.000000,2,6177.0,399.0,150.0,0,150.0,0,...,20,24.0,10,1,1,0,1,1,0,1
285886,3272886.0,7008212.0,29.990000,4,7900.0,345.0,150.0,2,224.0,1,...,22,24.0,19,2,2,2,2,2,1,4
104256,3091256.0,2071522.0,107.949997,4,11690.0,111.0,150.0,4,226.0,0,...,22,24.0,19,2,2,2,2,2,1,4
507860,3494860.0,13299752.0,241.949997,4,2616.0,327.0,150.0,1,102.0,0,...,22,24.0,19,2,2,2,2,2,1,4
196382,3183382.0,4412283.0,117.000000,4,13780.0,298.0,150.0,4,226.0,1,...,22,24.0,19,2,2,2,2,2,1,4


## 8. Guardado de artefactos y datasets limpios (uso interno, previo a feature engineering)

In [9]:
X_train = reduce_mem_usage(X_train)
X_test = reduce_mem_usage(X_test)

X_train["isFraud"] = y_train.values
X_test["isFraud"] = y_test.values

X_train.to_parquet(PROC_DIR / "train_clean.parquet", index=False)
X_test.to_parquet(PROC_DIR / "test_clean.parquet", index=False)

joblib.dump(cols_to_drop, ART_DIR / "cols_dropped_missing.joblib")
joblib.dump(num_imputer, ART_DIR / "num_imputer.joblib")
joblib.dump(frequent_categories, ART_DIR / "frequent_categories.joblib")
joblib.dump(encoder, ART_DIR / "ordinal_encoder.joblib")
joblib.dump({"num_cols": num_cols, "cat_cols": cat_cols}, ART_DIR / "col_types.joblib")

print("Guardado: data/processed/train_clean.parquet", X_train.shape)
print("Guardado: data/processed/test_clean.parquet ", X_test.shape)


Memoria: 1468.78 MB -> 723.13 MB (reducción del 50.8%)


Memoria: 367.20 MB -> 180.78 MB (reducción del 50.8%)


Guardado: data/processed/train_clean.parquet (472432, 422)
Guardado: data/processed/test_clean.parquet  (118108, 422)


## 9. Resumen

- Split 80/20 estratificado por `isFraud`, hecho antes de cualquier imputación (`random_state=42`).
- Se eliminaron las columnas con >90% de missing en train (mismo listado aplicado a test).
- Imputación: mediana (numéricas, ajustada en train) / `"missing"` explícito (categóricas).
- Categorías con <0.1% de frecuencia en train (o ausentes en train) agrupadas como `"rare"`.
- Encoding ordinal ajustado solo en train, con manejo explícito de categorías desconocidas.
- Resultado guardado en `data/processed/train_clean.parquet` / `test_clean.parquet`, listo para
  `03_feature_engineering.ipynb`.